In [1]:
!pip install datasets

In [2]:
#Loading the dataset
from datasets import load_dataset
ds = load_dataset("knkarthick/dialogsum")

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [7]:
ds['train'][1]['dialogue']

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [8]:
ds['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

In [9]:
!pip install transformers

In [10]:
#Testing BART Model without finetuning sumarrization task
from transformers import pipeline
pipe = pipeline("summarization", model = "facebook/bart-large-cnn")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [11]:
article_1 = ds['train'][1]['dialogue']

In [12]:
#BART Model give summary
pipe(article_1, max_length = 20, min_length = 10, do_sample = False)

[{'summary_text': 'Ricky has received his Polio, Tetanus and Hepatitis B shots.'}]

In [13]:
# Original Summary given in the dataset
ds['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

In [14]:
#Finetuning the model for text summarization

In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [18]:
#tokenization
def preprocess_function(batch):
  source = batch['dialogue']
  target = batch['summary']

  source_ids = tokenizer(source, truncation = True, padding = 'max_length', max_length = 128)
  target_ids = tokenizer(target, truncation = True, padding = 'max_length', max_length = 128)

  labels = target_ids['input_ids']
  labels = [[(label if label != tokenizer.pad_token_id else -100) for label in label_example] for label_example in labels]

  return {
      "input_ids": source_ids['input_ids'],
      "attention_mask": source_ids['attention_mask'],
      "labels": labels
  }


In [19]:
df_source = ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [22]:
#training_arguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = "/content",
    per_device_train_batch_size = 8,
    num_train_epochs = 2,
    remove_unused_columns = True
)

In [24]:
#Trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = df_source['train'],
    eval_dataset = df_source['test']
)

In [25]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jasweertadikonda (jasweertadikonda-northern-arizona-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,1.591600
1000,1.489800
1500,1.434800
2000,1.081900
2500,1.019300
3000,1.001600


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3917: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=3116, training_loss=1.2610011241555368, metrics={'train_runtime': 3116.1466, 'train_samples_per_second': 7.997, 'train_steps_per_second': 1.0, 'total_flos': 6750530835578880.0, 'train_loss': 1.2610011241555368, 'epoch': 2.0})

In [26]:
#Saving the model
model.save_pretrained('/content/Model_dir')
tokenizer.save_pretrained('/content/Model_dir')

('/content/Model_dir/tokenizer_config.json',
 '/content/Model_dir/special_tokens_map.json',
 '/content/Model_dir/vocab.json',
 '/content/Model_dir/merges.txt',
 '/content/Model_dir/added_tokens.json',
 '/content/Model_dir/tokenizer.json')

In [27]:
!zip -r model_dir.zip /content/Model_dir

  adding: content/Model_dir/ (stored 0%)
  adding: content/Model_dir/tokenizer_config.json (deflated 75%)
  adding: content/Model_dir/special_tokens_map.json (deflated 52%)
  adding: content/Model_dir/merges.txt (deflated 53%)
  adding: content/Model_dir/tokenizer.json (deflated 82%)
  adding: content/Model_dir/config.json (deflated 62%)
  adding: content/Model_dir/generation_config.json (deflated 47%)
  adding: content/Model_dir/vocab.json (deflated 59%)
  adding: content/Model_dir/model.safetensors (deflated 7%)
